# 五科題目與答案核對

此 Notebook 檢查整合輸出中的信託篩選結果、四科官方 PDF 題庫比較、離線包對照及入口連結。資料來源均為整合目錄內複製檔；執行只讀取並驗證，不修改來源題庫。

信託篩選條件：答案須有唯一明確官方答案、題目完整且可計分、題庫答案與官方答案一致；多答案疑義與送分題排除。

In [2]:
from pathlib import Path
import base64, json, re
root = Path(r"C:\Users\ehack\OneDrive\文件\Agent\證照考試整合資料")
reports = root / "07_核對報告"
trust = json.loads((reports / "trust_audit_details.json").read_text(encoding="utf-8"))
four = json.loads((reports / "four_subject_audit_details.json").read_text(encoding="utf-8"))
packages = json.loads((reports / "offline_package_audit.json").read_text(encoding="utf-8"))
print("Loaded audit records:", len(trust["curated_question_bank"]["source_missing_official_question_keys"]), "missing trust keys;")
print("four-subject PDF rows:", four["cleaned_official_pdf_questions"], "; offline packs:", len(packages))

Loaded audit records: 8 missing trust keys;
four-subject PDF rows: 7484 ; offline packs: 5


In [3]:
curated = trust["curated_question_bank"]
assert curated["candidate_records"] == 2852
assert curated["included_records"] == 2852 and curated["excluded_records"] == 0
assert curated["answer_mismatches"] == []
assert len(curated["source_missing_official_question_keys"]) == 8
assert sum(x["classification"] == "ambiguous_or_invalid" for x in curated["source_missing_official_question_keys"]) == 2
assert sum(x["classification"] == "sendoff" for x in curated["source_missing_official_question_keys"]) == 1
output = root / curated["delivered_file"]
wrapper = output.read_text(encoding="utf-8")
encoded = re.search(r"questions:\s*'([^']+)'", wrapper, re.S).group(1)
page = base64.b64decode(encoded).decode("utf-8")
rows = json.loads(re.search(r"const RAW_QUESTIONS\s*=\s*(\[.*?\]);", page, re.S).group(1))
assert len(rows) == 2852 and sum(row["p"] == 47 for row in rows) == 130
assert next(row["a"] for row in rows if (row["p"], row["s"], row["n"]) == (53, 2, 53)) == "2"
assert curated["embedded_output_recheck"]["matches_audited_payload"] is True
print("Trust included:", len(rows), "; period 47:", sum(row["p"] == 47 for row in rows), "; answer mismatches: 0")

Trust included: 2852 ; period 47: 130 ; answer mismatches: 0


In [4]:
assert four["raw_official_pdf_questions"] == 7781
assert four["cleaned_official_pdf_questions"] == 7484
pdf_html = four["pdf_html_comparison"]
assert pdf_html["expected_records"] == pdf_html["actual_records"] == 7484
for field in ("missing_keys", "extra_keys", "answer_difference_keys", "content_difference_keys", "occurrence_count_difference_keys"):
    assert pdf_html[field] == [], field
assert four["question_pack_total"] == 7483
pack_diff = four["pdf_pack_comparison"]
assert pack_diff["missing_keys"] == [] and pack_diff["extra_keys"] == []
assert len(pack_diff["occurrence_count_difference_keys"]) == 1
assert all(not p["missing_ids"] and not p["extra_ids"] and not p["answer_differences"] and not p["question_or_option_differences"] for p in packages)
print("PDF-extracted questions match merged bank exactly:", pdf_html["actual_records"], "; pack rows:", four["question_pack_total"], "; duplicate occurrence differences:", len(pack_diff["occurrence_count_difference_keys"]))

PDF-extracted questions match merged bank exactly: 7484 ; pack rows: 7483 ; duplicate occurrence differences: 1


In [5]:
index = (root / "index.html").read_text(encoding="utf-8")
hrefs = re.findall(r'href="([^"]+)"', index)
missing_links = [href for href in hrefs if not (root / href).exists()]
assert not missing_links, missing_links
assert len(hrefs) == 8, hrefs
print("Relative index links resolved:", len(hrefs), "; missing:", len(missing_links))

Relative index links resolved: 8 ; missing: 0
